# In this notebook we will clean the dataset

Setting of PySpark

In [34]:
# setting up the pyspark enivronment 
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("NYC Taxi Trip Duration Prediction").getOrCreate()

Loading the raw dataset 


In [35]:
taxi_df=spark.read.csv("../data/raw/train.csv",header=True, inferSchema=True)

In [36]:
taxi_df.show(5)

+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|       id|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|  pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|store_and_fwd_flag|trip_duration|
+---------+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|id2875421|        2|2016-03-14 17:24:55|2016-03-14 17:32:30|              1| -73.9821548461914| 40.76793670654297|-73.96463012695312|40.765602111816406|                 N|          455|
|id2377394|        1|2016-06-12 00:43:35|2016-06-12 00:54:38|              1|-73.98041534423828|40.738563537597656|-73.99948120117188| 40.73115158081055|                 N|          663|
|id3858529|        2|2016-01-19 11:35:24|2016-01-19 12:10:48|    

In [37]:
drop_taxi_df=taxi_df.drop("id")

In [38]:
drop_taxi_df.show(5)

+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|  pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|store_and_fwd_flag|trip_duration|
+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|        2|2016-03-14 17:24:55|2016-03-14 17:32:30|              1| -73.9821548461914| 40.76793670654297|-73.96463012695312|40.765602111816406|                 N|          455|
|        1|2016-06-12 00:43:35|2016-06-12 00:54:38|              1|-73.98041534423828|40.738563537597656|-73.99948120117188| 40.73115158081055|                 N|          663|
|        2|2016-01-19 11:35:24|2016-01-19 12:10:48|              1| -73.9790267944336|40.763938903808594|-74.005332

Passenger count validation

In [39]:
from pyspark.sql.functions import col
values_to_remove=[0,7,8,9]
filtered_taxi_df=drop_taxi_df.filter(~col("passenger_count").isin(values_to_remove))

In [40]:
filtered_taxi_df.show(5)

+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|  pickup_longitude|   pickup_latitude| dropoff_longitude|  dropoff_latitude|store_and_fwd_flag|trip_duration|
+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+------------------+------------------+-------------+
|        2|2016-03-14 17:24:55|2016-03-14 17:32:30|              1| -73.9821548461914| 40.76793670654297|-73.96463012695312|40.765602111816406|                 N|          455|
|        1|2016-06-12 00:43:35|2016-06-12 00:54:38|              1|-73.98041534423828|40.738563537597656|-73.99948120117188| 40.73115158081055|                 N|          663|
|        2|2016-01-19 11:35:24|2016-01-19 12:10:48|              1| -73.9790267944336|40.763938903808594|-74.005332

Trip duration filtering 

In [41]:
from pyspark.sql.functions import col, approx_percentile
percentiles = filtered_taxi_df.approxQuantile("trip_duration", [0.25, 0.5, 0.75], 0.01)
print("25th Percentile:", percentiles[0])
print("50th Percentile:", percentiles[1])
print("75th Percentile:", percentiles[2])

25th Percentile: 391.0
50th Percentile: 660.0
75th Percentile: 1069.0


In [42]:
# computing the interquartile range (IQR)
iqr=percentiles[2]-percentiles[0]
print("Interquartile Range (IQR):", iqr)

Interquartile Range (IQR): 678.0


In [43]:
lower_bound=percentiles[0]-1.5*iqr
upper_bound=percentiles[2]+1.5*iqr
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

Lower Bound: -626.0
Upper Bound: 2086.0


In [44]:
from pyspark.sql.functions import col
filtered_taxi_df=filtered_taxi_df.filter((col("trip_duration") >=60) & (col("trip_duration") <=7200))

In [45]:
filtered_taxi_df.select("trip_duration").describe().show()

+-------+-----------------+
|summary|    trip_duration|
+-------+-----------------+
|  count|          1447777|
|   mean| 840.874750047832|
| stddev|653.2580634225803|
|    min|               60|
|    max|             7191|
+-------+-----------------+



Coordinate validation


In [46]:
filtered_taxi_df.select("pickup_longitude","pickup_latitude","dropoff_longitude","dropoff_latitude").describe().show()

+-------+-------------------+--------------------+-------------------+--------------------+
|summary|   pickup_longitude|     pickup_latitude|  dropoff_longitude|    dropoff_latitude|
+-------+-------------------+--------------------+-------------------+--------------------+
|  count|            1447777|             1447777|            1447777|             1447777|
|   mean| -73.97362587804326|  40.750974436693376| -73.97354524578182|  40.751851777718024|
| stddev| 0.0707645754382266|0.032670039763363225|0.07050394938945528|0.035724914202075644|
|    min|-121.93334197998047|   34.35969543457031|-121.93330383300781|    32.1811408996582|
|    max| -61.33552932739258|   51.88108444213867| -61.33552932739258|   43.92102813720703|
+-------+-------------------+--------------------+-------------------+--------------------+



In [47]:
cleaned_taxi_df=filtered_taxi_df.filter(col('pickup_longitude').between(-75,-73) &
                                        col('pickup_latitude').between(40,42) &
                                        col('dropoff_longitude').between(-75,-73) &
                                        col('dropoff_latitude').between(40,42))

In [48]:
cleaned_taxi_df.select("pickup_longitude","pickup_latitude","dropoff_longitude","dropoff_latitude").describe().show()

+-------+--------------------+--------------------+-------------------+-------------------+
|summary|    pickup_longitude|     pickup_latitude|  dropoff_longitude|   dropoff_latitude|
+-------+--------------------+--------------------+-------------------+-------------------+
|  count|             1447736|             1447736|            1447736|            1447736|
|   mean|   -73.9735693362617|   40.75099431543353| -73.97347000839427|  40.75187905588459|
| stddev|0.037925501189683665|0.027972805958210594|0.03578884359656852|0.03227912871484297|
|    min|  -74.72671508789062|  40.099788665771484| -74.77542877197266| 40.153743743896484|
|    max|  -73.09227752685547|   41.69679641723633| -73.04737854003906|  41.69335174560547|
+-------+--------------------+--------------------+-------------------+-------------------+



In [49]:
clean_df=cleaned_taxi_df.drop_duplicates()

In [50]:
clean_df.show(5)

+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+-----------------+------------------+-------------+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|  pickup_longitude|   pickup_latitude| dropoff_longitude| dropoff_latitude|store_and_fwd_flag|trip_duration|
+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+-----------------+------------------+-------------+
|        1|2016-06-02 06:12:57|2016-06-02 06:19:23|              1| -73.9904556274414|  40.7559814453125|-73.98375701904297|40.75518035888672|                 N|          386|
|        1|2016-05-15 18:19:54|2016-05-15 18:24:18|              2| -73.9518051147461| 40.78193664550781|-73.94275665283203| 40.7860221862793|                 N|          264|
|        2|2016-04-04 15:51:28|2016-04-04 15:54:02|              2|-73.98506164550781|40.724056243896484|-73.97767639160

Datetime validation

In [51]:
from pyspark.sql.functions import col, year, min as spark_min, max as spark_max

# 1. Check null timestamps
print("Null pickup_datetime rows:", clean_df.filter(col("pickup_datetime").isNull()).count())
print("Null dropoff_datetime rows:", clean_df.filter(col("dropoff_datetime").isNull()).count())

Null pickup_datetime rows: 0
Null dropoff_datetime rows: 0


In [ ]:
clean_df.filter(
    col("pickup_datetime").isNull() | col("dropoff_datetime").isNull()
).show(5, truncate=False)


+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+
|vendor_id|pickup_datetime|dropoff_datetime|passenger_count|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|store_and_fwd_flag|trip_duration|
+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+
+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+



In [53]:
 #3. Check impossible trips: pickup after dropoff
print("Rows where pickup_datetime > dropoff_datetime:",
      clean_df.filter(col("pickup_datetime") > col("dropoff_datetime")).count())

Rows where pickup_datetime > dropoff_datetime: 0


In [54]:
clean_df.filter(
    col("pickup_datetime") > col("dropoff_datetime")
).show(5, truncate=False)

+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+
|vendor_id|pickup_datetime|dropoff_datetime|passenger_count|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|store_and_fwd_flag|trip_duration|
+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+
+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+



In [55]:
cleaned_df=clean_df.filter(col("pickup_datetime")<=col("dropoff_datetime"))

In [56]:
# 5. Check unrealistic year values
print("Rows with pickup year outside 2016:",
      cleaned_df.filter(year(col("pickup_datetime")) != 2016).count())

print("Rows with dropoff year outside 2016:",
      cleaned_df.filter(year(col("dropoff_datetime")) != 2016).count())

Rows with pickup year outside 2016: 0
Rows with dropoff year outside 2016: 0


In [57]:
cleaned_df.filter(
    (year(col("pickup_datetime")) != 2016) |
    (year(col("dropoff_datetime")) != 2016)
).show(5, truncate=False)

+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+
|vendor_id|pickup_datetime|dropoff_datetime|passenger_count|pickup_longitude|pickup_latitude|dropoff_longitude|dropoff_latitude|store_and_fwd_flag|trip_duration|
+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+
+---------+---------------+----------------+---------------+----------------+---------------+-----------------+----------------+------------------+-------------+



In [58]:
# 6. Keep only valid 2016 timestamps
cleaned_df = cleaned_df.filter(
    (col("pickup_datetime") >= "2016-01-01 00:00:00") &
    (col("pickup_datetime") <= "2016-12-31 23:59:59") &
    (col("dropoff_datetime") >= "2016-01-01 00:00:00") &
    (col("dropoff_datetime") <= "2016-12-31 23:59:59")
)

In [59]:
# 7. Final validation: check min and max timestamps
cleaned_df.select(
    spark_min("pickup_datetime").alias("min_pickup_datetime"),
    spark_max("pickup_datetime").alias("max_pickup_datetime"),
    spark_min("dropoff_datetime").alias("min_dropoff_datetime"),
    spark_max("dropoff_datetime").alias("max_dropoff_datetime")
).show(truncate=False)

+-------------------+-------------------+--------------------+--------------------+
|min_pickup_datetime|max_pickup_datetime|min_dropoff_datetime|max_dropoff_datetime|
+-------------------+-------------------+--------------------+--------------------+
|2016-01-01 00:00:17|2016-06-30 23:59:39|2016-01-01 00:03:31 |2016-07-01 00:48:20 |
+-------------------+-------------------+--------------------+--------------------+



In [60]:
cleaned_df.show(5)

+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+-----------------+------------------+-------------+
|vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|  pickup_longitude|   pickup_latitude| dropoff_longitude| dropoff_latitude|store_and_fwd_flag|trip_duration|
+---------+-------------------+-------------------+---------------+------------------+------------------+------------------+-----------------+------------------+-------------+
|        1|2016-06-02 06:12:57|2016-06-02 06:19:23|              1| -73.9904556274414|  40.7559814453125|-73.98375701904297|40.75518035888672|                 N|          386|
|        1|2016-05-15 18:19:54|2016-05-15 18:24:18|              2| -73.9518051147461| 40.78193664550781|-73.94275665283203| 40.7860221862793|                 N|          264|
|        2|2016-04-04 15:51:28|2016-04-04 15:54:02|              2|-73.98506164550781|40.724056243896484|-73.97767639160

removed dataleakage feature 

In [61]:
cleaned_df=cleaned_df.drop("dropoff_datetime")

In [62]:
cleaned_df.printSchema()

root
 |-- vendor_id: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- trip_duration: integer (nullable = true)



In [63]:
cleaned_df.show(5)

+---------+-------------------+---------------+------------------+------------------+------------------+-----------------+------------------+-------------+
|vendor_id|    pickup_datetime|passenger_count|  pickup_longitude|   pickup_latitude| dropoff_longitude| dropoff_latitude|store_and_fwd_flag|trip_duration|
+---------+-------------------+---------------+------------------+------------------+------------------+-----------------+------------------+-------------+
|        1|2016-06-02 06:12:57|              1| -73.9904556274414|  40.7559814453125|-73.98375701904297|40.75518035888672|                 N|          386|
|        1|2016-05-15 18:19:54|              2| -73.9518051147461| 40.78193664550781|-73.94275665283203| 40.7860221862793|                 N|          264|
|        2|2016-04-04 15:51:28|              2|-73.98506164550781|40.724056243896484|-73.97767639160156|40.72130584716797|                 N|          154|
|        2|2016-04-30 19:23:20|              1|-73.9860229492187

Dataset size verification

In [64]:
# check the rows before cleaning the data 
taxi_df.count()

1458644

In [65]:
# check the rows after cleaning the data
cleaned_df.count()

1447731

Save cleaned dataset

In [66]:
cleaned_df.write.mode("overwrite").option("header", "true").csv("../data/processed/cleaned_taxi_data.csv")